# Melanoma Detection — Full Training on the Real ISIC Dataset

Run this notebook in **Google Colab** (Runtime → Change runtime type → GPU).

This trains the exact same `MelanomaClassifier` (ResNet18 transfer learning) used in the project's demo, but on the **full ISIC dataset** instead of the small synthetic sample.

**Steps:**
1. Install dependencies
2. Download the ISIC dataset
3. Train
4. Evaluate with real metrics (accuracy, precision, recall, F1, AUC, confusion matrix)
5. Export `model_weights.pth` to plug into the Flask backend

In [ ]:
!pip install -q torch torchvision scikit-learn kaggle

## 1. Get the ISIC dataset

Easiest route: use the Kaggle mirror of the ISIC 2019/2020 datasets.

1. Get a Kaggle API token (kaggle.com -> Account -> Create New API Token) -> downloads `kaggle.json`
2. Upload it in the next cell when prompted

In [ ]:
from google.colab import files
uploaded = files.upload()  # upload your kaggle.json here

import os
os.makedirs('/root/.kaggle', exist_ok=True)
!cp kaggle.json /root/.kaggle/
!chmod 600 /root/.kaggle/kaggle.json

# ISIC 2020 challenge dataset (resized JPEGs, more manageable size)
!kaggle datasets download -d nroman/melanoma-external-malignant-256
!unzip -q melanoma-external-malignant-256.zip -d isic_data

## 2. Organize into ImageFolder structure

`torchvision.datasets.ImageFolder` expects:
```
isic_data/
    benign/*.jpg
    malignant/*.jpg
```
Adjust this cell based on the actual folder/CSV structure of the dataset you downloaded (ISIC datasets vary — some come with a metadata CSV mapping image_id -> label instead of folders).

In [ ]:
import pandas as pd, shutil, os

# Example for CSV-labeled datasets — adapt column names to your chosen dataset's metadata file
# df = pd.read_csv('isic_data/train.csv')
# os.makedirs('isic_data/organized/benign', exist_ok=True)
# os.makedirs('isic_data/organized/malignant', exist_ok=True)
# for _, row in df.iterrows():
#     src = f"isic_data/train/{row['image_name']}.jpg"
#     dst_folder = 'malignant' if row['target'] == 1 else 'benign'
#     shutil.copy(src, f"isic_data/organized/{dst_folder}/{row['image_name']}.jpg")

DATA_DIR = 'isic_data/organized'  # point this at your final folder structure

## 3. Model definition (identical to `model/model.py` in the main project)

In [ ]:
import torch
import torch.nn as nn
from torchvision import models, transforms, datasets
from torch.utils.data import DataLoader, random_split

class MelanomaClassifier(nn.Module):
    def __init__(self, pretrained=True, freeze_backbone=False):
        super().__init__()
        weights = models.ResNet18_Weights.DEFAULT if pretrained else None
        self.backbone = models.resnet18(weights=weights)
        if freeze_backbone:
            for p in self.backbone.parameters():
                p.requires_grad = False
        in_features = self.backbone.fc.in_features
        self.backbone.fc = nn.Sequential(
            nn.Linear(in_features, 128), nn.ReLU(), nn.Dropout(0.3), nn.Linear(128, 2)
        )
    def forward(self, x):
        return self.backbone(x)

## 4. Handle class imbalance

Real ISIC data is heavily imbalanced (malignant cases are a small minority — often <2%). Use a `WeightedRandomSampler` and/or class-weighted loss, otherwise the model will just predict "benign" for everything and still show high accuracy while being clinically useless.

In [ ]:
from torch.utils.data import WeightedRandomSampler
import numpy as np

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
])

full_dataset = datasets.ImageFolder(DATA_DIR, transform=train_transform)
class_names = full_dataset.classes
print('Classes:', class_names)

targets = np.array(full_dataset.targets)
class_counts = np.bincount(targets)
class_weights = 1.0 / class_counts
sample_weights = class_weights[targets]
sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)

val_size = int(0.15 * len(full_dataset))
train_size = len(full_dataset) - val_size
train_ds, val_ds = random_split(full_dataset, [train_size, val_size])

train_loader = DataLoader(train_ds, batch_size=32, sampler=sampler)
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False)

## 5. Train

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

model = MelanomaClassifier(pretrained=True).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', patience=2)

EPOCHS = 15
best_val_acc = 0.0

for epoch in range(EPOCHS):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * imgs.size(0)
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    train_acc = correct / total

    model.eval()
    val_correct, val_total = 0, 0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = model(imgs)
            _, preds = torch.max(outputs, 1)
            val_correct += (preds == labels).sum().item()
            val_total += labels.size(0)
    val_acc = val_correct / val_total
    scheduler.step(val_acc)

    print(f'Epoch {epoch+1}/{EPOCHS} | loss={running_loss/total:.4f} | train_acc={train_acc:.3f} | val_acc={val_acc:.3f}')

    if val_acc >= best_val_acc:
        best_val_acc = val_acc
        torch.save({'model_state_dict': model.state_dict(), 'class_names': class_names}, 'model_weights.pth')

print('Best val accuracy:', best_val_acc)

## 6. Proper evaluation (accuracy alone is misleading on imbalanced medical data)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import torch.nn.functional as F

model.eval()
all_preds, all_labels, all_probs = [], [], []
with torch.no_grad():
    for imgs, labels in val_loader:
        imgs = imgs.to(device)
        outputs = model(imgs)
        probs = F.softmax(outputs, dim=1)[:, 1].cpu().numpy()
        preds = torch.argmax(outputs, dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())
        all_probs.extend(probs)

print(classification_report(all_labels, all_preds, target_names=class_names))
print('Confusion matrix:\n', confusion_matrix(all_labels, all_preds))
print('ROC-AUC:', roc_auc_score(all_labels, all_probs))

## 7. Download the trained weights, then drop into `model/model_weights.pth` in the main project to power the real backend.

In [ ]:
from google.colab import files
files.download('model_weights.pth')